# Recurrent Neural Networks
## for sentiment analysis

Source:
- Keras.io documentation
- "Sentiment Analysis using Keras & LSTM" @ Medium

Title: Bidirectional LSTM on IMDB

Description: Train a 2-layer bidirectional LSTM on the IMDB movie review sentiment classification dataset.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

In [2]:
from keras import Input
from keras.datasets import imdb
from keras.layers import BatchNormalization, Dense, Dropout, Embedding, Flatten, LSTM
from keras.models import Sequential
from keras.optimizers import Adam, SGD, RMSprop
from keras.regularizers import l1, l2
from keras.utils import plot_model, to_categorical, pad_sequences

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Setup

In [3]:
top_words = 5000  # Only consider the top 5k words
input_length = 200  # Only consider the first 200 words of each movie review

In [4]:
def plot_history(train_hist):
    pd.DataFrame(train_hist).plot(figsize=(8, 5))
    plt.grid(True)
    plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
    plt.show()

### Loading dataset

In [5]:
## Load the IMDB movie review sentiment data
# Load the data, already split between train and test sets
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=top_words)

In [6]:
print(f"Train size: {x_train.shape[0]}")
print(f"Test size:  {x_test.shape[0]}")

Train size: 25000
Test size:  25000


Labels:
- 0 (negative review) or
- 1 (positive review).

In [7]:
np.unique(y_train)

array([0, 1])

In [8]:
REVIEW_INDEX = 2

print(x_train[REVIEW_INDEX])

[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 2, 54, 61, 369, 13, 71, 149, 14, 22, 112, 4, 2401, 311, 12, 16, 3711, 33, 75, 43, 1829, 296, 4, 86, 320, 35, 534, 19, 263, 4821, 1301, 4, 1873, 33, 89, 78, 12, 66, 16, 4, 360, 7, 4, 58, 316, 334, 11, 4, 1716, 43, 645, 662, 8, 257, 85, 1200, 42, 1228, 2578, 83, 68, 3912, 15, 36, 165, 1539, 278, 36, 69, 2, 780, 8, 106, 14, 2, 1338, 18, 6, 22, 12, 215, 28, 610, 40, 6, 87, 326, 23, 2300, 21, 23, 22, 12, 272, 40, 57, 31, 11, 4, 22, 47, 6, 2307, 51, 9, 170, 23, 595, 116, 595, 1352, 13, 191, 79, 638, 89, 2, 14, 9, 8, 106, 607, 624, 35, 534, 6, 227, 7, 129, 113]


These are vector representations of word indexes. We need to pad the sequence to a maximum of 500 words. For that Keras provides us with the pad_sequences method:



In [9]:
# Use pad_sequence to standardize sequence length:
# this will truncate sequences longer than 200 words and zero-pad sequences shorter than 200 words.
x_train = pad_sequences(x_train, maxlen=input_length)
x_test = pad_sequences(x_test, maxlen=input_length)

In [10]:
print(x_train[REVIEW_INDEX])

[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    1   14   47    8   30   31    7    4  249  108    7
    4    2   54   61  369   13   71  149   14   22  112    4 2401  311
   12   16 3711   33   75   43 1829  296    4   86  320   35  534   19
  263 4821 1301    4 1873   33   89   78   12   66   16    4  360    7
    4   58  316  334   11    4 1716   43  645  662    8  257   85 1200
   42 1228 2578   83   68 3912   15   36  165 1539  278   36   69    2
  780    8  106   14    2 1338   18    6   22   12  215   28  610   40
    6   87  326   23 2300   21   23   22   12  272   40   57   31   11
    4   22   47    6 2307   51    9  170   23  595  116  595 1352   13
  191   79  638   89    2   14    9    8  106  607  624   35  534    6
  227 

Now we need to build a word_to_id dictionary so that these indexes can be transformed into words for further analysis. In the dictionary, we will provide PAD token to index 0, START token to index 1, and UNK token to index 2. So we have to shift the default indexes by 3 to adjust these tokens.

In [11]:
word_to_id = imdb.get_word_index()
word_to_id = {k:(v+3) for k,v in word_to_id.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2

After building word_to_it we need to id_to_word:

In [12]:
id_to_word = {idx:word for word, idx in word_to_id.items()}

**Convertion examples**:

In [13]:
id_to_word[20]

'movie'

In [14]:
decoded_text = [id_to_word[id] for id in x_train[REVIEW_INDEX]]

print(" ".join(decoded_text))

<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <START> this has to be one of the worst films of the <UNK> when my friends i were watching this film being the target audience it was aimed at we just sat watched the first half an hour with our jaws touching the floor at how bad it really was the rest of the time everyone else in the theatre just started talking to each other leaving or generally crying into their popcorn that they actually paid money they had <UNK> working to watch this <UNK> excuse for a film it must have looked like a great idea on paper but on film it looks like no one in the film has a clue what is going on crap acting crap costumes i can't get across how <UNK> this

### Build the model

In [15]:
embedding_vector_length = 32 

In [ ]:
### Create a LSTM model for sentiment (binary) classification
### YOUR CODE HERE:

model = Sequential([
    Input((...,)),
    Embedding(top_words, embedding_vector_length),
    ...
    ...
    ...
]) 

In [ ]:
model.summary()

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) 

In [ ]:
epochs = 6
batch_size = 64

trained = model.fit(x_train, y_train, 
                    validation_data=(x_test, y_test),
                    epochs=epochs, batch_size=batch_size)

### Evaluate the model

In [ ]:
plot_history(trained.history)

### Predict sentiment

In [17]:
REVIEW_INDEX = 1

In [ ]:
decoded_text = [id_to_word[id] for id in x_train[REVIEW_INDEX]]

print(" ".join(decoded_text))

In [ ]:
my_review = np.array([x_train[REVIEW_INDEX]])

my_review

In [ ]:
### Classify this review (`my_review`) in terms of sentiment
### YOUR CODE HERE:

...

In [ ]:
### Given now the following review text, transform this input and predict sentiment

my_review_text = "One of the finest films made in recent years."

my_review_vec = []
for word in my_review_text.split(" "):
    if word[-1] == ".":
        word = word[:-1]
    my_review_vec.append(word_to_id[str.lower(word)])

my_review_vec = pad_sequences([my_review_vec], input_length)


### YOUR CODE HERE:

...